In [1]:
import pickle
import pandas as pd
import numpy as np

In [ ]:
belief_col = "Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol"

In [ ]:
for idx, window in enumerate(time_windows):
    print(f"[{idx+1}/{len(time_windows)}] Processing predictions for window: {window}")
    
    # Load the fitted model for this window
    model_dir = f"{base_dir}/{belief_col}_{window}_stan_output"
    fit_path = f"{model_dir}/fit.pkl"
    
    if not os.path.exists(fit_path):
        print(f"  Warning: fit.pkl not found for window {window}")
        continue
    
    with open(fit_path, "rb") as f:
            fit = pickle.load(f)
            break

In [4]:
strat_frame = pd.read_csv("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/cleaned_sf.csv")

In [5]:
strat_frame

,Unnamed: 0,sex_idx,age_idx,eth_idx,income_idx,state_idx,vote_idx,education_idx,N_star.1,N_star.2,...,LP_pct.2020.votes.G,LP_pct.2020.votes.stay.home,LP_pct.2020.votes.OTHER,LP_pct.2022.votes.D,LP_pct.2022.votes.R,LP_pct.2022.votes.L,LP_pct.2022.votes.G,LP_pct.2022.votes.stay.home,LP_pct.2022.votes.OTHER,state_abbreviation_upper
0,0,2,1,4,1,1,1,1,1.867455,0.548105,...,-0.997116,-0.154947,2.071398,0.286556,-0.196120,0.136231,-0.175703,-0.514353,-0.467282,AK
1,1,2,1,4,1,1,2,1,0.013752,0.139760,...,-0.997116,-0.154947,2.071398,0.286556,-0.196120,0.136231,-0.175703,-0.514353,-0.467282,AK
2,2,2,1,4,1,1,3,1,0.264992,1.602773,...,-0.997116,-0.154947,2.071398,0.286556,-0.196120,0.136231,-0.175703,-0.514353,-0.467282,AK
3,3,2,1,4,1,1,4,1,0.001838,0.007018,...,-0.997116,-0.154947,2.071398,0.286556,-0.196120,0.136231,-0.175703,-0.514353,-0.467282,AK
4,4,2,1,4,1,1,5,1,0.000000,0.000000,...,-0.997116,-0.154947,2.071398,0.286556,-0.196120,0.136231,-0.175703,-0.514353,-0.467282,AK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207793,207793,1,6,1,5,51,4,2,0.158298,0.012458,...,-0.997116,0.314307,2.264716,-1.463526,1.104902,0.441713,-0.175703,0.042198,1.215613,WY
207794,207794,1,6,1,5,51,5,1,0.000000,0.000000,...,-0.997116,0.314307,2.264716,-1.463526,1.104902,0.441713,-0.175703,0.042198,1.215613,WY
207795,207795,1,6,1,5,51,5,2,0.000000,0.000000,...,-0.997116,0.314307,2.264716,-1.463526,1.104902,0.441713,-0.175703,0.042198,1.215613,WY
207796,207796,1,6,1,5,51,6,1,0.078727,0.064580,...,-0.997116,0.314307,2.264716,-1.463526,1.104902,0.441713,-0.175703,0.042198,1.215613,WY


In [5]:
with open(f"/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/Dropped_ICAR_MrP/{belief_col}_stan_output/fit.pkl", "rb") as f:
    fit = pickle.load(f)

/home/yuyue/stan_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def generate_cell_level_predictions(fit,strat_frame, belief_col, batch_size = 10000):

    # Extract posterior samples
    kappa = fit.stan_variable('kappa') # 2D array (draws x K-1)
    eta_sex_star = fit.stan_variable('eta_sex_star') # 2D array (draws x categories)
    eta_age_star = fit.stan_variable('eta_age_star')
    eta_eth_star = fit.stan_variable('eta_eth_star')
    eta_vote_star = fit.stan_variable('eta_vote_star')
    eta_income_star = fit.stan_variable('eta_income_star')
    eta_state = fit.stan_variable('eta_state')
    mu_state_LP = fit.stan_variable('mu_state_LP')  # 1D array (draws,)

    n_draws = kappa.shape[0]
    n_cells = len(strat_frame)
    K = kappa.shape[1] + 1  

    # Initialize arrays
    pred_mean = np.zeros(n_cells)
    pred_median = np.zeros(n_cells)
    pred_lower = np.zeros(n_cells)
    pred_upper = np.zeros(n_cells)
    pred_sd = np.zeros(n_cells)
    prob_cats_mean = np.zeros((n_cells, K))
    prob_cats_median = np.zeros((n_cells, K))
    prob_cats_lower = np.zeros((n_cells, K))
    prob_cats_upper = np.zeros((n_cells, K))
    prob_cats_sd = np.zeros((n_cells, K))

    all_draws = np.zeros((n_cells, n_draws,))
    
    # For national aggregation - accumulate weighted probabilities
    n_star_cols = [c for c in strat_frame.columns if c.startswith('N_star.') and c != 'N_star.median']
    national_cat_accum = {n_col: np.zeros((n_draws, K)) for n_col in n_star_cols}

    n_batches = int(np.ceil(n_cells / batch_size))
    print(f"\nGenerating predictions for {n_cells:,} cells in {n_batches} batches of size {batch_size}")
    for batch_idx in range(n_batches):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, n_cells)
        batch_n = end_idx - start_idx
        batch_data = strat_frame.iloc[start_idx:end_idx]

        mu_batch = np.zeros((batch_n, n_draws))

        for i, (_, row) in enumerate(batch_data.iterrows()):
            mu_batch[i, :] = (      # process one cell at a time
                eta_sex_star[:, int(row['sex_idx'])-1] + # select all draws(rows) and the column of corresponding category
                eta_age_star[:, int(row['age_idx'])-1] +
                eta_vote_star[:, int(row['vote_idx'])-1] +
                eta_eth_star[:, int(row['eth_idx'])-1] +
                eta_income_star[:, int(row['income_idx'])-1] +
                eta_state[:, int(row['state_idx'])-1]+
                mu_state_LP[:, int(row['state_idx'])-1] 
            )

        # calculate probabilities for each category
        prob_batch = np.zeros((batch_n, n_draws, K))

        for k in range(K):
            if k == 0:
                prob_batch[:, :, k] = 1 / (1 + np.exp(-(kappa[:, 0] - mu_batch)))
            elif k == K - 1:
                prob_batch[:, :, k] = 1- 1/(
                    (1 + np.exp(- (kappa[:, K-2] - mu_batch)))
                )
            else:
                prob_batch[:, :, k] = (
                    1/(1+np.exp(-(kappa[:, k] - mu_batch))) -
                    1/(1+np.exp(-(kappa[:, k-1] - mu_batch)))
                )
        
        # calculate expected value
        categories = np.arange(1, K+1)
        expected_y_batch = np.sum(prob_batch * categories[np.newaxis, np.newaxis, :], axis=2)

        # summary stats for expected value
        pred_mean[start_idx:end_idx] = np.mean(expected_y_batch, axis=1)
        pred_median[start_idx:end_idx] = np.median(expected_y_batch, axis=1)
        pred_lower[start_idx:end_idx] = np.percentile(expected_y_batch, 2.5, axis=1)
        pred_upper[start_idx:end_idx] = np.percentile(expected_y_batch, 97.5, axis=1)
        pred_sd[start_idx:end_idx]  = np.std(expected_y_batch, axis=1)  

        # summary stats for each category probability
        for k in range(K):
            prob_cats_mean[start_idx:end_idx, k] = np.mean(prob_batch[:, :, k], axis=1)
            prob_cats_median[start_idx:end_idx, k] = np.median(prob_batch[:, :, k], axis=1)
            prob_cats_lower[start_idx:end_idx, k] = np.percentile(prob_batch[:, :, k], 2.5, axis=1)
            prob_cats_upper[start_idx:end_idx, k] = np.percentile(prob_batch[:, :, k], 97.5, axis=1)
            prob_cats_sd[start_idx:end_idx, k]  = np.std(prob_batch[:, :, k], axis=1)
        
        all_draws[start_idx:end_idx, :] = expected_y_batch
        
        # Accumulate weighted category probabilities for national level
        for n_col in n_star_cols:
            weights_raw = batch_data[n_col].values
            weights_raw = np.nan_to_num(weights_raw, nan=0.0)
            total_weight = strat_frame[n_col].sum()  # Total weight across all cells
            weights_normalized = weights_raw / total_weight
            
            # Add weighted contribution of this batch to national totals
            for k in range(K):
                national_cat_accum[n_col][:, k] += np.sum(
                    prob_batch[:, :, k] * weights_normalized[:, np.newaxis], 
                    axis=0
                )

    # Add results summary stats to strat_frame
    strat_frame[f'{belief_col}_pred_mean'] = pred_mean
    strat_frame[f'{belief_col}_pred_median'] = pred_median
    strat_frame[f'{belief_col}_pred_lower'] = pred_lower
    strat_frame[f'{belief_col}_pred_upper'] = pred_upper
    strat_frame[f'{belief_col}_pred_sd'] = pred_sd
    for k in range(K):
        strat_frame[f'{belief_col}_prob_cat_mean_{k+1}'] = prob_cats_mean[:, k]
        strat_frame[f'{belief_col}_prob_cat_median_{k+1}'] = prob_cats_median[:, k]
        strat_frame[f'{belief_col}_prob_cat_lower_{k+1}'] = prob_cats_lower[:, k]
        strat_frame[f'{belief_col}_prob_cat_upper_{k+1}'] = prob_cats_upper[:, k]
        strat_frame[f'{belief_col}_prob_cat_sd_{k+1}'] = prob_cats_sd[:, k]
    
    # save all draws in a DataFrame
    draws_df = pd.DataFrame(all_draws, columns=[f'draw_{i+1}' for i in range(n_draws)])

    for col in ['state_simple', 'sex_idx', 'age_idx', 'eth_idx', 'income_idx']:
        if col in strat_frame.columns:
            draws_df[col] = strat_frame[col].values
        else:
            print(f"Column {col} not found in strat_frame.")

    print(f"\nCell-level predictions complete")
    return strat_frame, all_draws, draws_df, national_cat_accum, K

In [7]:
def process_national_category_predictions(national_cat_accum, K):
    """Convert accumulated national category probabilities to summary statistics"""
    n_star_cols = list(national_cat_accum.keys())
    
    # Stack all weight samples: shape (n_weight_samples, n_draws, K)
    national_cat_draws = np.array([national_cat_accum[col] for col in n_star_cols])
    
    # Summary statistics for each category
    results = []
    for k in range(K):
        flat = national_cat_draws[:, :, k].flatten()
        results.append({
            'category': k + 1,
            'prob_mean': np.mean(flat),
            'prob_median': np.median(flat),
            'prob_lower': np.percentile(flat, 2.5),
            'prob_upper': np.percentile(flat, 97.5),
            'prob_sd': np.std(flat)
        })
    
    return pd.DataFrame(results), national_cat_draws

In [2]:
import pandas as pd

In [6]:
strat_frame = pd.read_csv(f"/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/Dropped_ICAR_MrP/{belief_col}_stan_output/cell_level_predictions.csv")

In [13]:
strat_frame

,Unnamed: 0,sex_idx,age_idx,eth_idx,income_idx,state_idx,vote_idx,N_star.1,N_star.2,N_star.3,...,Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_prob_cat_mean_2,Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_prob_cat_median_2,Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_prob_cat_lower_2,Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_prob_cat_upper_2,Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_prob_cat_sd_2,Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_prob_cat_mean_3,Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_prob_cat_median_3,Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_prob_cat_lower_3,Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_prob_cat_upper_3,Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_prob_cat_sd_3
0,0,2,1,4,1,1,1,1.867455,0.548105,1.565072,...,0.164748,0.156727,0.069849,0.305814,0.060553,0.010054,0.009079,0.003640,0.022099,0.004922
1,1,2,1,4,1,1,2,0.013752,0.139760,0.207898,...,0.482415,0.489443,0.298715,0.625427,0.085376,0.056206,0.051776,0.021032,0.117843,0.025400
2,2,2,1,4,1,1,3,0.264992,1.602773,0.317292,...,0.066054,0.061156,0.025106,0.135946,0.029208,0.003497,0.003169,0.001248,0.007786,0.001755
3,3,2,1,4,1,1,4,0.001838,0.007018,0.013940,...,0.492695,0.512571,0.230939,0.646442,0.115542,0.068962,0.057833,0.014862,0.185846,0.046401
4,4,2,1,4,1,1,5,0.000000,0.000000,0.000000,...,0.048172,0.033551,0.001867,0.177150,0.048391,0.002621,0.001684,0.000088,0.010734,0.003123
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110155,110155,1,6,1,5,51,2,23.309498,26.801340,33.842180,...,0.592364,0.595294,0.532653,0.636210,0.026591,0.093374,0.092337,0.063249,0.129435,0.016833
110156,110156,1,6,1,5,51,3,17.174077,19.340563,9.853957,...,0.108805,0.107416,0.073597,0.151731,0.019887,0.005983,0.005869,0.003783,0.008864,0.001294
110157,110157,1,6,1,5,51,4,0.102195,0.023442,0.284320,...,0.585035,0.605818,0.414744,0.654033,0.064839,0.113269,0.101263,0.036431,0.256810,0.057606
110158,110158,1,6,1,5,51,5,0.000000,0.000000,0.000000,...,0.078907,0.061072,0.004077,0.258464,0.067027,0.004516,0.003150,0.000200,0.017432,0.004566


In [11]:
draws_df = pd.read_csv(f"/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/Dropped_ICAR_MrP/{belief_col}_stan_output/cell_level_all_draws.csv")

In [12]:
draws_df

,draw_1,draw_2,draw_3,draw_4,draw_5,draw_6,draw_7,draw_8,draw_9,draw_10,...,draw_7996,draw_7997,draw_7998,draw_7999,draw_8000,state_simple,sex_idx,age_idx,eth_idx,income_idx
0,1.087525,1.299538,1.103956,1.315462,1.095009,1.250161,1.142078,1.282541,1.217170,1.125129,...,1.205357,1.177644,1.090275,1.274858,1.108960,Alaska,2,1,4,1
1,1.383758,1.777101,1.424134,1.848830,1.427461,1.759688,1.538204,1.767743,1.719263,1.499121,...,1.666953,1.612674,1.392173,1.790800,1.449139,Alaska,2,1,4,1
2,1.035297,1.104462,1.037090,1.123417,1.042612,1.106739,1.055596,1.104671,1.098595,1.043092,...,1.098254,1.072423,1.030273,1.117541,1.041617,Alaska,2,1,4,1
3,1.680095,1.717990,1.543375,1.791077,1.591479,1.788955,1.464359,1.601150,1.860828,1.598196,...,1.596858,1.660586,1.527983,2.036816,1.279600,Alaska,2,1,4,1
4,1.004207,1.032980,1.046974,1.046247,1.014982,1.076769,1.040493,1.108717,1.014526,1.038279,...,1.015099,1.088945,1.017078,1.029322,1.011472,Alaska,2,1,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110155,1.741305,1.773356,1.682515,1.833696,1.625236,1.823727,1.719421,1.829940,1.669100,1.774409,...,1.790141,1.795474,1.777089,1.794112,1.729614,Wyoming,1,6,1,5
110156,1.113529,1.103277,1.086048,1.117874,1.081441,1.129423,1.097768,1.126062,1.084684,1.103012,...,1.142243,1.126704,1.105597,1.118686,1.101912,Wyoming,1,6,1,5
110157,2.045580,1.714220,1.809979,1.775791,1.798374,1.852695,1.641438,1.663568,1.811315,1.874569,...,1.720304,1.842739,1.923696,2.040077,1.522724,Wyoming,1,6,1,5
110158,1.014469,1.032579,1.107666,1.044015,1.029329,1.093679,1.071994,1.130828,1.012331,1.092087,...,1.022702,1.153772,1.061448,1.029633,1.029300,Wyoming,1,6,1,5


In [8]:
print(strat_frame.columns.to_list())

['Unnamed: 0', 'sex_idx', 'age_idx', 'eth_idx', 'income_idx', 'state_idx', 'vote_idx', 'N_star.1', 'N_star.2', 'N_star.3', 'N_star.4', 'N_star.5', 'N_star.6', 'N_star.7', 'N_star.8', 'N_star.9', 'N_star.10', 'N_star.11', 'N_star.12', 'N_star.13', 'N_star.14', 'N_star.15', 'N_star.16', 'N_star.17', 'N_star.18', 'N_star.19', 'N_star.20', 'N_star.21', 'N_star.22', 'N_star.23', 'N_star.24', 'N_star.25', 'N_star.26', 'N_star.27', 'N_star.28', 'N_star.29', 'N_star.30', 'N_star.31', 'N_star.32', 'N_star.33', 'N_star.34', 'N_star.35', 'N_star.36', 'N_star.37', 'N_star.38', 'N_star.39', 'N_star.40', 'N_star.41', 'N_star.42', 'N_star.43', 'N_star.44', 'N_star.45', 'N_star.46', 'N_star.47', 'N_star.48', 'N_star.49', 'N_star.50', 'N_star.51', 'N_star.52', 'N_star.53', 'N_star.54', 'N_star.55', 'N_star.56', 'N_star.57', 'N_star.58', 'N_star.59', 'N_star.60', 'N_star.61', 'N_star.62', 'N_star.63', 'N_star.64', 'N_star.65', 'N_star.66', 'N_star.67', 'N_star.68', 'N_star.69', 'N_star.70', 'N_star.71',

In [8]:
strat_frame, all_draws, draws_df, national_cat_accum, K = generate_cell_level_predictions(fit, strat_frame, belief_col, batch_size=10000)

# shape strat_frame: (n_cells, demographic columns + prediction summary columns)
strat_frame.to_csv(f"/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/Dropped_ICAR_MrP/{belief_col}_stan_output/cell_level_predictions.csv", index=False)

# shape draws_df: (n_cells, n_draws + demographic columns)
draws_df.to_csv(f"/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/Dropped_ICAR_MrP/{belief_col}_stan_output/cell_level_all_draws.csv", index=False)

# Process national category probabilities
national_cat_preds, national_cat_draws = process_national_category_predictions(national_cat_accum, K)
national_cat_preds.to_csv(f"/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/Dropped_ICAR_MrP/{belief_col}_stan_output/national_category_probabilities.csv", index=False)

print(f"National category predictions shape: {national_cat_preds.shape}")
national_cat_preds


Generating predictions for 110,160 cells in 12 batches of size 10000

Cell-level predictions complete
National category predictions shape: (3, 6)


,category,prob_mean,prob_median,prob_lower,prob_upper,prob_sd
0,1,0.644329,0.644373,0.628816,0.659282,0.007741
1,2,0.314461,0.314390,0.301073,0.328197,0.006910
2,3,0.041210,0.041144,0.036742,0.045842,0.002337


In [8]:
# get state level predictions with all draws and weight samples
def state_level_predictions(strat_frame, all_draws, state_col):
    n_cells, n_draws = all_draws.shape
    n_star_cols = [c for c in strat_frame.columns if c.startswith('N_star.') and c!= 'N_star.median']

    results = []
    all_state_draws = {}
    states = strat_frame[state_col].unique()
    for state in states:
        state_mask = strat_frame[state_col] == state #boolean series of True/False
        state_rows = np.where(state_mask)[0]  #indices of rows belonging to the state
        state_draws = []

        for n_col in n_star_cols:
            weights_raw = strat_frame.loc[state_mask, n_col].values  #raw weights for one weight sample of the state
            weights_raw = np.nan_to_num(weights_raw, nan=0.0)
            weights = weights_raw / weights_raw.sum()  #normalize weights
            weighted_draws = np.average(all_draws[state_rows, :], axis=0, weights=weights)  #weighted average collapsed for states for each draw
            state_draws.append(weighted_draws) #draws for one weight sample

        state_draws = np.array(state_draws)  #shape: (n_weight_samples, n_draws)

        flat = state_draws.flatten()  #flatten results to 1D array for summary stats
        results.append({
            state_col: state,
            'pred_mean': np.mean(flat),
            'pred_median': np.median(flat),
            'pred_lower': np.percentile(flat, 2.5),
            'pred_upper': np.percentile(flat, 97.5),
            'pred_sd': np.std(flat),
            'n_cells': state_mask.sum(),
            'n_draws': n_draws,
            'n_weight_samples': len(n_star_cols)
        })
        all_state_draws[state] = state_draws
    
    # convert all_state_draws to DataFrame
    rows = []
    for state, arr in all_state_draws.items(): # arr shape: (n_weight_samples, n_draws)
        df_state = pd.DataFrame(arr, columns=[f'draw_{i+1}' for i in range(n_draws)]) # for each state create a dataframe of draws with rows for each weight sample and columns for each draw
        df_state.insert(0, state_col, state)  # add state column
        rows.append(df_state)
    df_all = pd.concat(rows, ignore_index=True) # concatenate all state dataframes one under another

    return pd.DataFrame(results), df_all

In [9]:
state_preds, state_all_draws = state_level_predictions(strat_frame, all_draws, state_col='state_idx')

# shape state_preds: (n_states, summary columns)
state_preds.to_csv(f"/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/Dropped_ICAR_MrP/{belief_col}_stan_output/state_level_predictions.csv", index=False)
# shape state_all_draws: (n_states * n_weight_samples, n_draws) -- state one under another
state_all_draws.to_csv(f"/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/Dropped_ICAR_MrP/{belief_col}_stan_output/state_level_all_draws.csv", index=False)

In [10]:
state_preds

,state_idx,pred_mean,pred_median,pred_lower,pred_upper,pred_sd,n_cells,n_draws,n_weight_samples
0,1,1.427304,1.423100,1.274436,1.607801,0.082856,2160,8000,104
1,2,1.494282,1.489629,1.423808,1.587394,0.041846,2160,8000,104
2,3,1.459254,1.458721,1.392751,1.531801,0.035003,2160,8000,104
3,4,1.403680,1.404251,1.351088,1.454712,0.026163,2160,8000,104
4,5,1.291032,1.290946,1.261110,1.321111,0.015410,2160,8000,104
5,6,1.400261,1.399889,1.352975,1.449948,0.024404,2160,8000,104
6,7,1.345668,1.346717,1.267218,1.420314,0.038403,2160,8000,104
7,8,1.110066,1.108938,1.072457,1.152800,0.020567,2160,8000,104
8,9,1.359508,1.358250,1.296746,1.430903,0.033331,2160,8000,104
9,10,1.431102,1.431406,1.392727,1.469741,0.019716,2160,8000,104


In [11]:
def national_level_predictions(strat_frame, all_draws):
    n_cells, n_draws = all_draws.shape
    n_star_cols = [c for c in strat_frame.columns if c.startswith('N_star.') and c!= 'N_star.median']

    nat_list = []
    for n_col in n_star_cols:
        weights_raw = strat_frame[n_col].values  #raw weights for one weight sample
        weights_raw = np.nan_to_num(weights_raw, nan=0.0)
        weights = weights_raw / weights_raw.sum()  #normalize weights
        weighted_draws = np.average(all_draws, axis=0, weights=weights)  #weighted average collapsed for each draw
        nat_list.append(weighted_draws) #draws for one weight sample
    national_draws = np.array(nat_list)  # shape: (n_weight_samples, n_draws)
    df_national_draws = pd.DataFrame(national_draws, columns=[f'draw_{i+1}' for i in range(n_draws)])

    flat = national_draws.flatten()  #flatten results to 1D array for summary stats
    summary_stats = {
            'pred_mean': np.mean(flat),
            'pred_median': np.median(flat),
            'pred_lower': np.percentile(flat, 2.5),
            'pred_upper': np.percentile(flat, 97.5),
            'pred_sd': np.std(flat),
            'n_cells': n_cells,
            'n_draws': n_draws,
            'n_weight_samples': len(n_star_cols)
        }
    return pd.DataFrame([summary_stats]), df_national_draws

In [12]:
national_preds, df_national_draws = national_level_predictions(strat_frame, all_draws)

# shape national_preds: one row with summary columns
national_preds.to_csv(f"/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/Dropped_ICAR_MrP/{belief_col}_stan_output/national_level_predictions.csv", index=False)
# shape national_all_draws: (n_weight_samples, n_draws)
df_national_draws.to_csv(f"/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/Dropped_ICAR_MrP/{belief_col}_stan_output/national_level_all_draws.csv", index=False)

In [13]:
df_national_draws

,draw_1,draw_2,draw_3,draw_4,draw_5,draw_6,draw_7,draw_8,draw_9,draw_10,...,draw_7991,draw_7992,draw_7993,draw_7994,draw_7995,draw_7996,draw_7997,draw_7998,draw_7999,draw_8000
0,1.388471,1.391048,1.396686,1.392018,1.371561,1.405126,1.391913,1.403521,1.382732,1.391156,...,1.393039,1.418737,1.402180,1.410168,1.419159,1.401820,1.385870,1.403789,1.409029,1.392608
1,1.388860,1.391515,1.397085,1.392129,1.372517,1.405298,1.392059,1.403712,1.383127,1.391768,...,1.393256,1.419103,1.402294,1.410597,1.419498,1.402154,1.386255,1.404212,1.409292,1.392897
2,1.388496,1.390960,1.396779,1.391976,1.371621,1.405219,1.392064,1.403598,1.382851,1.391013,...,1.393098,1.418721,1.402271,1.410116,1.419224,1.401889,1.385793,1.403777,1.409170,1.392540
3,1.388647,1.391237,1.397064,1.392181,1.371990,1.405520,1.392180,1.403785,1.383001,1.391562,...,1.393320,1.418947,1.402553,1.410336,1.419398,1.402053,1.386134,1.404158,1.409297,1.392837
4,1.388578,1.391323,1.397113,1.391728,1.372265,1.405020,1.392243,1.403578,1.383174,1.391463,...,1.393252,1.418918,1.402394,1.410416,1.419407,1.402124,1.386247,1.403983,1.409099,1.392932
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,1.388416,1.391009,1.396764,1.392047,1.371561,1.405209,1.391975,1.403563,1.382810,1.391007,...,1.393124,1.418588,1.402349,1.410099,1.419306,1.401891,1.385727,1.403623,1.409109,1.392506
100,1.388554,1.391173,1.397080,1.391991,1.371905,1.405178,1.392104,1.403626,1.382956,1.391360,...,1.393196,1.418812,1.402387,1.410294,1.419283,1.402048,1.386029,1.404042,1.409180,1.392759
101,1.388884,1.391427,1.397246,1.392454,1.372249,1.405736,1.392408,1.403857,1.383292,1.391577,...,1.393415,1.419079,1.402761,1.410517,1.419546,1.402169,1.386226,1.404262,1.409462,1.392979
102,1.389150,1.391413,1.397257,1.392425,1.371866,1.405599,1.392420,1.404057,1.383365,1.391573,...,1.393494,1.419205,1.402682,1.410562,1.419488,1.402206,1.386330,1.404365,1.409517,1.393137


In [14]:
national_preds

,pred_mean,pred_median,pred_lower,pred_upper,pred_sd,n_cells,n_draws,n_weight_samples
0,1.39688,1.396789,1.379165,1.415235,0.009111,110160,8000,104
